# SmartGear Retail Medallion Data Pipeline

## Project Overview

This project implements a modern retail analytics data platform using Databricks Community Edition and Apache Spark.

The solution is designed to simulate a real-world enterprise retail environment where both batch and streaming sales data are processed for analytical reporting and business intelligence.

The project follows the Medallion Architecture approach consisting of:

- Bronze Layer → Raw data ingestion
- Silver Layer → Cleaned and transformed data
- Gold Layer → Business-ready analytical datasets

---

## Business Objective

The primary business objectives of this project are:

- Analyze retail sales performance
- Monitor regional revenue trends
- Identify top-performing products
- Evaluate store-level performance
- Enable scalable analytics architecture

---

## Technologies Used

- Databricks Community Edition
- Apache Spark
- Delta Tables
- Python (PySpark)
- Apache Kafka
- Confluent Cloud
- Medallion Architecture
- Azure Data Engineering Concepts

---

## Architecture Approach

The project follows a layered architecture approach:

### Bronze Layer
Stores raw ingested retail transaction data without modification.

### Silver Layer
Performs data cleansing, standardization, and transformation.

### Gold Layer
Generates aggregated business KPIs and analytical datasets for reporting and dashboards.

In [0]:
print("SmartGear Medallion Pipeline Started")

SmartGear Medallion Pipeline Started


# Bronze Layer - Raw Data Ingestion

## Objective

The Bronze Layer is responsible for ingesting raw retail sales data into the data platform without applying business transformations.

This layer acts as the source-of-truth storage layer and preserves the original structure of incoming datasets.

---

## Business Significance

The Bronze Layer is important because:

- Preserves historical raw data
- Enables auditability and traceability
- Supports reprocessing in case of downstream failures
- Acts as the foundation for all transformations

In enterprise systems, the Bronze Layer commonly receives data from:

- ERP systems
- POS systems
- Streaming platforms such as Kafka
- Cloud storage systems
- APIs and transactional databases

---

## Dataset Description

The dataset contains retail transaction records including:

- Order ID
- Order Date
- Store ID
- Product Name
- Quantity Sold
- Unit Price
- Region

The data is ingested into a Delta Table named:

`shopstream.default.bronze_sales`

---

## Expected Outcome

After ingestion:

- Raw sales data becomes queryable
- Spark transformations can be applied
- Downstream Silver and Gold layers can be generated

In [0]:
from pyspark.sql.functions import *

# Read Bronze Layer Table
bronze_df = spark.table("shopstream.default.bronze_sales")

# Display Bronze Data
display(bronze_df)

OrderID,OrderDate,Region,StoreID,Product,Quantity,UnitPrice
1001,2025-03-14,East,115,Headphones,3,81.65
1002,2025-02-19,West,110,Smartwatch,3,242.5
1003,2025-03-17,West,113,Printer,2,152.59
1004,2025-01-05,West,118,Camera,4,463.4
1005,2025-01-07,West,110,Tablet,3,370.01
1006,2025-03-22,East,113,Smartphone,4,639.47
1007,2025-02-19,North,118,Drone,4,747.21
1008,2025-02-21,West,112,Printer,2,151.06
1009,2025-03-30,North,117,Smartphone,1,614.89
1010,2025-01-30,West,108,Tablet,3,415.43


# Silver Layer - Data Cleaning and Transformation

## Objective

The Silver Layer transforms raw Bronze data into a clean, standardized, and analytics-ready dataset.

This layer applies business rules, data validation, cleansing operations, and schema standardization.

---

## Business Significance

The Silver Layer improves data quality and reliability before analytics are performed.

This layer helps organizations:

- Remove inconsistent records
- Standardize formats
- Improve reporting accuracy
- Enable trusted analytics
- Prepare datasets for KPI generation

In enterprise environments, the Silver Layer is commonly used for:

- Data validation
- Null handling
- Data enrichment
- Type conversion
- Deduplication

---

## Transformation Logic

The following transformations are applied:

- Revenue column calculation
- Standardized schema formatting
- Derived business metrics
- Analytical feature preparation

---

## Expected Outcome

The transformed dataset will be stored as:

`shopstream.default.silver_sales`

This dataset will serve as the foundation for Gold Layer business analytics.

In [0]:
# Read Bronze Layer
silver_df = spark.table("shopstream.default.bronze_kafka_sales")

# Create Revenue Column
silver_df = silver_df.withColumn(
    "Revenue",
    col("quantity") * col("price")
)

# Display Silver Data
display(silver_df)

# Save Silver Layer
silver_df.write.mode("overwrite").saveAsTable(
    "shopstream.default.silver_sales"
)

order_id,timestamp,store_id,product,quantity,price,region,ingestion_time,source,Revenue
2002,2025-04-01 10:02:20,102,Smartphone,1,650.75,South,2026-05-15T13:51:18.693Z,kafka_stream,650.75
2003,2025-04-01 10:03:05,103,Tablet,3,320.4,East,2026-05-15T13:51:18.693Z,kafka_stream,961.1999999999999
2010,2025-04-01 10:10:15,110,Smartwatch,3,210.3,East,2026-05-15T13:51:18.693Z,kafka_stream,630.9000000000001
2001,2025-04-01 10:01:15,101,Laptop,2,850.5,North,2026-05-15T13:51:18.693Z,kafka_stream,1701.0
2007,2025-04-01 10:07:55,107,Printer,1,190.0,South,2026-05-15T13:51:18.693Z,kafka_stream,190.0
2008,2025-04-01 10:08:30,108,Gaming Console,2,499.99,West,2026-05-15T13:51:18.693Z,kafka_stream,999.98
2004,2025-04-01 10:04:10,104,Headphones,4,120.99,West,2026-05-15T13:51:18.693Z,kafka_stream,483.96
2005,2025-04-01 10:05:25,105,Camera,1,540.0,North,2026-05-15T13:51:18.693Z,kafka_stream,540.0
2006,2025-04-01 10:06:40,106,Monitor,2,280.15,East,2026-05-15T13:51:18.693Z,kafka_stream,560.3
2009,2025-04-01 10:09:45,109,Drone,1,799.5,North,2026-05-15T13:51:18.693Z,kafka_stream,799.5


# Gold Layer - Business KPI Aggregation

## Objective

The Gold Layer contains business-ready aggregated datasets optimized for reporting, dashboarding, and executive decision-making.

This layer converts cleaned Silver data into high-value analytical KPIs.

---

## Business Significance

The Gold Layer enables management teams to analyze:

- Regional revenue trends
- Product performance
- Store efficiency
- Sales contribution
- Revenue distribution

This layer supports:

- Executive dashboards
- Business intelligence reporting
- Operational monitoring
- Strategic planning
- Revenue optimization

---

## KPI Calculations

The following KPIs are generated:

1. Total Revenue by Region
2. Top Selling Products
3. Store Performance Metrics

---

## Expected Gold Tables

The following analytical tables will be created:

- `shopstream.default.gold_region_kpi`
- `shopstream.default.gold_top_products`
- `shopstream.default.gold_store_performance`

These tables are optimized for dashboards and business reporting.

In [0]:
from pyspark.sql.functions import sum, desc

# Read Silver Layer
gold_df = spark.table("shopstream.default.silver_sales")

# Region Revenue KPI
region_kpi = gold_df.groupBy("region").agg(
    sum("Revenue").alias("Total_Revenue")
)

# Display
display(region_kpi)

# Save Gold Table
region_kpi.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_region_kpi"
)

Region,Total_Revenue
South,840.75
East,2152.3999999999996
North,3040.5
West,1483.94


In [0]:
# Product Performance KPI
top_products = gold_df.groupBy("Product").agg(
    sum("Revenue").alias("Product_Revenue")
).orderBy(desc("Product_Revenue"))

# Display
display(top_products)

# Save Gold Table
top_products.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_top_products"
)

Product,Product_Revenue
Laptop,1701.0
Gaming Console,999.98
Tablet,961.1999999999999
Drone,799.5
Smartphone,650.75
Smartwatch,630.9000000000001
Monitor,560.3
Camera,540.0
Headphones,483.96
Printer,190.0


In [0]:
# Store Performance KPI
store_performance = gold_df.groupBy("store_id").agg(
    sum("revenue").alias("Store_Revenue")
).orderBy(desc("Store_Revenue"))

# Display
display(store_performance)

# Save Gold Table
store_performance.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_store_performance"
)

store_id,Store_Revenue
101,1701.0
108,999.98
103,961.1999999999999
109,799.5
102,650.75
110,630.9000000000001
106,560.3
105,540.0
104,483.96
107,190.0


B4 – Kafka Streaming Integration with Databricks
Objective

The objective of this implementation is to establish a real-time streaming pipeline between Confluent Cloud Kafka and Databricks using Spark Structured Streaming.

This pipeline continuously ingests streaming retail transaction data and prepares it for Medallion Architecture processing.

Technologies Used
Apache Kafka (Confluent Cloud)
Spark Structured Streaming
Databricks Community Edition
PySpark
Delta Lake
Streaming Workflow
Kafka producer continuously generates retail transactions.
Events are pushed into the smartgear_orders Kafka topic.
Databricks connects securely to Kafka using SASL_SSL authentication.
Spark Structured Streaming consumes real-time messages.
JSON messages are parsed using a predefined schema.
Structured records are prepared for Bronze layer ingestion.
Business Significance

This architecture enables:

Real-time sales monitoring
Live operational analytics
Low-latency retail insights
Scalable distributed event processing
Foundation for enterprise streaming pipelines
Features Implemented
Kafka Topic Subscription
Secure Authentication using API Keys
Real-Time Streaming Ingestion
JSON Parsing
Schema Enforcement
Structured Streaming Pipeline

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Kafka Schema
sales_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("timestamp", StringType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("region", StringType(), True)
])

In [0]:
KAFKA_BOOTSTRAP = "pkc-xrnwx.asia-south2.gcp.confluent.cloud:9092"
API_KEY = "7XHH3PL6L2222QGY"
API_SECRET = "cflt9bdQzBtFKjdbkJnaPgb1Upiz2f3JJzlSAx2B1WaPVOwMYJsNILj7H7qRrC1w"
TOPIC_NAME = "smartgear_orders"

In [0]:
streaming_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", TOPIC_NAME)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        f'org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'
    )
    .option("startingOffsets", "earliest")
    .load()
)

In [0]:
KAFKA_BOOTSTRAP = "pkc-xrnwx.asia-south2.gcp.confluent.cloud:9092"
API_KEY = "7XHH3PL6L2222QGY"
API_SECRET = "cflt9bdQzBtFKjdbkJnaPgb1Upiz2f3JJzlSAx2B1WaPVOwMYJsNILj7H7qRrC1w"
TOPIC_NAME = "smartgear_orders"

In [0]:
streaming_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", TOPIC_NAME)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'
    )
    .option("startingOffsets", "earliest")
    .load()
)

In [0]:
print(streaming_df)

DataFrame[key: binary, value: binary, topic: string, partition: int, offset: bigint, timestamp: timestamp, timestampType: int]


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

sales_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("timestamp", StringType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("region", StringType(), True)
])

parsed_df = (
    streaming_df
    .selectExpr("CAST(value AS STRING)")
    .select(
        from_json(col("value"), sales_schema).alias("data")
    )
    .select("data.*")
)

In [0]:
print(parsed_df)

DataFrame[order_id: int, timestamp: string, store_id: int, product: string, quantity: int, price: double, region: string]


Kafka Schema Definition and JSON Parsing
Objective

Kafka messages are received in JSON format.
To convert raw streaming events into structured analytical records, a schema is defined using PySpark StructType.

The schema ensures:

Data consistency
Type validation
Structured parsing
Streaming reliability
Fields Included
Field	Description
order_id	Unique order identifier
timestamp	Event generation timestamp
store_id	Retail store identifier
product	Product purchased
quantity	Quantity sold
price	Product price
region	Sales region

C1 – Bronze Layer Implementation
Objective

The Bronze layer is the raw ingestion layer of the Medallion Architecture.

Its purpose is to:

Capture raw streaming data from Kafka
Preserve original events
Enable replay and auditing
Provide fault-tolerant ingestion
Bronze Layer Characteristics
Append-only storage
Raw event preservation
Minimal transformations
High scalability
Reliable streaming ingestion
Metadata Added
Column	Purpose
ingestion_time	Timestamp when data entered the lakehouse
source	Identifies source system
Technologies Used
Apache Kafka
Spark Structured Streaming
Delta Lake
Databricks
Business Importance

Bronze layer acts as:

Enterprise raw data archive
Recovery point for failures
Foundation for Silver and Gold transformations
Historical audit repository

In [0]:
bronze_df = (
    parsed_df
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source", lit("kafka_stream"))
)

In [0]:
print(bronze_df)

DataFrame[order_id: int, timestamp: string, store_id: int, product: string, quantity: int, price: double, region: string, ingestion_time: timestamp, source: string]


In [0]:
bronze_query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/shopstream/default/checkpoints/bronze_checkpoint_v1"
    )
    .trigger(availableNow=True)
    .toTable("shopstream.default.bronze_kafka_sales")
)

C2 – Silver Layer Implementation
Objective

The Silver layer performs data cleansing, standardization, and validation on Bronze layer data before analytical processing.

Implementation is performed using Apache Spark PySpark DataFrame APIs inside Databricks.

Data Engineering Transformations
1. Null Handling

Critical null records are removed to maintain data quality and reporting accuracy.

2. Standardization

Region values are converted into uppercase standardized format for consistent aggregations.

3. Duplicate Removal

Duplicate records are removed based on:

order_id
timestamp
4. Revenue Calculation

A derived business metric is created:

Revenue = Quantity × Price

5. Data Type Validation

Correct Spark data types are enforced for reliable analytical processing.

Business Importance

The Silver layer improves:

Data consistency
Reporting reliability
Aggregation accuracy
Enterprise data governance

In [0]:
from pyspark.sql.functions import upper

silver_df = (
    bronze_df
    .dropna(subset=["order_id", "product", "price"])
    .dropDuplicates(["order_id", "timestamp"])
    .withColumn("region", initcap(col("region")))
    .withColumn("revenue", col("quantity") * col("price"))
)

In [0]:
print(silver_df)

DataFrame[order_id: int, timestamp: string, store_id: int, product: string, quantity: int, price: double, region: string, ingestion_time: timestamp, source: string, revenue: double]


In [0]:
dbutils.fs.rm(
    "/Volumes/shopstream/default/checkpoints/silver_temp_v1",
    True
)

False

In [0]:
silver_static_df = (
    silver_df.writeStream
    .format("memory")
    .queryName("silver_temp")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/shopstream/default/checkpoints/silver_temp_v2"
    )
    .start()
)

silver_static_df.awaitTermination()

In [0]:
final_silver_df = spark.sql("SELECT * FROM silver_temp")

In [0]:
display(final_silver_df)

order_id,timestamp,store_id,product,quantity,price,region,ingestion_time,source,revenue
2009,2025-04-01 10:09:45,109,Drone,1,799.5,North,2026-05-15T21:46:08.682Z,kafka_stream,799.5
2010,2025-04-01 10:10:15,110,Smartwatch,3,210.3,East,2026-05-15T21:46:08.682Z,kafka_stream,630.9000000000001
2004,2025-04-01 10:04:10,104,Headphones,4,120.99,West,2026-05-15T21:46:08.682Z,kafka_stream,483.96
2001,2025-04-01 10:01:15,101,Laptop,2,850.5,North,2026-05-15T21:46:08.682Z,kafka_stream,1701.0
2007,2025-04-01 10:07:55,107,Printer,1,190.0,South,2026-05-15T21:46:08.682Z,kafka_stream,190.0
2002,2025-04-01 10:02:20,102,Smartphone,1,650.75,South,2026-05-15T21:46:08.682Z,kafka_stream,650.75
2005,2025-04-01 10:05:25,105,Camera,1,540.0,North,2026-05-15T21:46:08.682Z,kafka_stream,540.0
2006,2025-04-01 10:06:40,106,Monitor,2,280.15,East,2026-05-15T21:46:08.682Z,kafka_stream,560.3
2003,2025-04-01 10:03:05,103,Tablet,3,320.4,East,2026-05-15T21:46:08.682Z,kafka_stream,961.1999999999999
2008,2025-04-01 10:08:30,108,Gaming Console,2,499.99,West,2026-05-15T21:46:08.682Z,kafka_stream,999.98


In [0]:
spark.sql("DROP TABLE IF EXISTS shopstream.default.silver_sales")

DataFrame[]

In [0]:
final_silver_df.write.mode("overwrite").saveAsTable(
    "shopstream.default.silver_sales"
)

# C2 – Analytical Transformations

## 1. Daily Revenue Trend

### Objective
Analyze revenue generated per day to identify sales trends over time.

### Business Significance
Daily revenue analysis helps:
- Monitor business growth
- Identify high-sales days
- Support forecasting and planning
- Detect abnormal revenue fluctuations

In [0]:
silver_data = spark.table("shopstream.default.silver_sales")

In [0]:
from pyspark.sql.functions import sum

daily_revenue = (
    silver_data
    .groupBy("timestamp")
    .agg(
        sum("Revenue").alias("daily_revenue")
    )
    .orderBy("timestamp")
)

In [0]:
display(daily_revenue)

timestamp,daily_revenue
2025-04-01 10:01:15,1701.0
2025-04-01 10:02:20,650.75
2025-04-01 10:03:05,961.1999999999999
2025-04-01 10:04:10,483.96
2025-04-01 10:05:25,540.0
2025-04-01 10:06:40,560.3
2025-04-01 10:07:55,190.0
2025-04-01 10:08:30,999.98
2025-04-01 10:09:45,799.5
2025-04-01 10:10:15,630.9000000000001


In [0]:
spark.sql("DROP TABLE IF EXISTS shopstream.default.gold_daily_revenue")

DataFrame[]

In [0]:
daily_revenue.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_daily_revenue"
)

# 2. Region-wise Contribution %

### Objective
Calculate percentage contribution of each region toward total business revenue.

### Business Significance
This analysis helps identify:
- High-performing regions
- Weak revenue zones
- Market expansion opportunities
- Regional business dependency

In [0]:
from pyspark.sql.functions import col

In [0]:
from pyspark.sql.functions import sum, round

total_revenue = silver_data.agg(
    sum("Revenue").alias("total")
).collect()[0]["total"]

region_contribution = (
    silver_data
    .groupBy("region")
    .agg(
        sum("Revenue").alias("region_revenue")
    )
    .withColumn(
        "contribution_percent",
        round(
            (col("region_revenue") / total_revenue) * 100,
            2
        )
    )
)

In [0]:
display(region_contribution)

region,region_revenue,contribution_percent
South,840.75,11.18
East,2152.3999999999996,28.63
North,3040.5,40.45
West,1483.94,19.74


In [0]:
region_contribution.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_region_contribution"
)

# 3. Top 3 Stores per Region

### Objective
Identify top-performing stores within each region based on revenue.

### Business Significance
This helps management:
- Identify best-performing stores
- Benchmark store performance
- Improve regional operations
- Support expansion planning

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

store_revenue = (
    silver_data
    .groupBy("region", "store_id")
    .agg(
        sum("Revenue").alias("store_revenue")
    )
)

window_spec = Window.partitionBy("region").orderBy(
    col("store_revenue").desc()
)

top_stores = (
    store_revenue
    .withColumn(
        "rank",
        dense_rank().over(window_spec)
    )
    .filter(col("rank") <= 3)
)

In [0]:
display(top_stores)

region,store_id,store_revenue,rank
East,103,961.1999999999999,1
East,110,630.9000000000001,2
East,106,560.3,3
North,101,1701.0,1
North,109,799.5,2
North,105,540.0,3
South,102,650.75,1
South,107,190.0,2
West,108,999.98,1
West,104,483.96,2


In [0]:
top_stores.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_top_stores_region"
)

# 4. Data Skew Identification

### Objective
Identify whether any region contributes disproportionately high revenue compared to others.

### Business Significance
Data skew analysis helps:
- Detect business dependency on one region
- Identify imbalance in sales distribution
- Improve load balancing and business diversification
- Support strategic decision-making

In [0]:
region_skew = (
    silver_data
    .groupBy("region")
    .agg(
        sum("Revenue").alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

In [0]:
display(region_skew)

region,total_revenue
North,3040.5
East,2152.3999999999996
West,1483.94
South,840.75


In [0]:
region_skew.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_data_skew"
)

# C3 – Gold Layer KPI Analytics

## 1. Total Revenue KPI

### Objective
Calculate total business revenue generated across all retail transactions.

### Business Significance
Total revenue is a core business KPI used for:
- Executive reporting
- Business growth tracking
- Financial monitoring
- Strategic planning

In [0]:
total_revenue_kpi = (
    silver_data
    .agg(
        sum("Revenue").alias("total_revenue")
    )
)

In [0]:
display(total_revenue_kpi)

total_revenue
7517.59


In [0]:
total_revenue_kpi.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_total_revenue_kpi"
)

# 2. Average Order Value KPI

### Objective
Calculate average revenue generated per order transaction.

### Business Significance
Average Order Value (AOV) helps:
- Measure customer purchasing behavior
- Evaluate pricing effectiveness
- Support upselling strategies
- Improve sales planning

In [0]:
from pyspark.sql.functions import avg

In [0]:
average_order_value = (
    silver_data
    .agg(
        avg("Revenue").alias("average_order_value")
    )
)

In [0]:
display(average_order_value)

average_order_value
751.759


In [0]:
average_order_value.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_average_order_value"
)

# 3. Top Selling Products KPI

### Objective
Identify products generating highest sales revenue.

### Business Significance
This helps:
- Optimize inventory planning
- Identify high-demand products
- Improve marketing focus
- Support product strategy decisions

In [0]:
top_products = (
    silver_data
    .groupBy("product")
    .agg(
        sum("Revenue").alias("product_revenue")
    )
    .orderBy(col("product_revenue").desc())
)

In [0]:
display(top_products)

product,product_revenue
Laptop,1701.0
Gaming Console,999.98
Tablet,961.1999999999999
Drone,799.5
Smartphone,650.75
Smartwatch,630.9000000000001
Monitor,560.3
Camera,540.0
Headphones,483.96
Printer,190.0


In [0]:
top_products.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_top_products"
)

# 4. Region-wise Sales KPI

### Objective
Analyze revenue distribution across regions.

### Business Significance
Region-wise sales analysis helps:
- Compare regional performance
- Identify strong and weak markets
- Support regional business planning
- Improve sales strategy allocation

In [0]:
region_sales_kpi = (
    silver_data
    .groupBy("region")
    .agg(
        sum("Revenue").alias("regional_revenue")
    )
    .orderBy(col("regional_revenue").desc())
)

In [0]:
display(region_sales_kpi)

region,regional_revenue
North,3040.5
East,2152.3999999999996
West,1483.94
South,840.75


In [0]:
region_sales_kpi.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_region_sales_kpi"
)

# 5. Store Performance KPI

### Objective
Evaluate revenue generated by each retail store.

### Business Significance
Store performance analysis helps:
- Compare store productivity
- Identify high-performing outlets
- Support operational improvements
- Optimize retail expansion planning

In [0]:
store_performance = (
    silver_data
    .groupBy("store_id")
    .agg(
        sum("Revenue").alias("store_revenue")
    )
    .orderBy(col("store_revenue").desc())
)

In [0]:
display(store_performance)

store_id,store_revenue
101,1701.0
108,999.98
103,961.1999999999999
109,799.5
102,650.75
110,630.9000000000001
106,560.3
105,540.0
104,483.96
107,190.0


In [0]:
spark.sql("DROP TABLE IF EXISTS shopstream.default.gold_store_performance")

DataFrame[]

In [0]:
store_performance.write.mode("overwrite").saveAsTable(
    "shopstream.default.gold_store_performance"
)

# C4 – Advanced SQL Analytics

## Objective
Perform advanced analytical reporting using Spark SQL on Silver layer data.

### Business Significance
SQL analytics enables:
- Business reporting
- Executive dashboards
- KPI querying
- Ad-hoc analytical insights

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW silver_sales_view AS
SELECT * FROM shopstream.default.silver_sales;

# 1. Total Revenue by Region

### Objective
Analyze total revenue generated by each region using SQL aggregation.

In [0]:
%sql
SELECT
    region,
    SUM(Revenue) AS total_revenue
FROM silver_sales_view
GROUP BY region
ORDER BY total_revenue DESC;

region,total_revenue
North,3040.5
East,2152.3999999999996
West,1483.94
South,840.75


# 2. Average Revenue Per Store

### Objective
Calculate average revenue generated per store using Spark SQL.

### Business Significance
This helps identify:
- Store efficiency
- Operational consistency
- Revenue balancing across outlets

In [0]:
%sql
SELECT
    store_id,
    AVG(Revenue) AS average_revenue
FROM silver_sales_view
GROUP BY store_id
ORDER BY average_revenue DESC

store_id,average_revenue
101,1701.0
108,999.98
103,961.1999999999999
109,799.5
102,650.75
110,630.9000000000001
106,560.3
105,540.0
104,483.96
107,190.0


# 3. Top Products by Revenue

### Objective
Identify products generating maximum business revenue.

### Business Significance
This analysis helps:
- Identify best-selling products
- Improve inventory planning
- Focus marketing efforts
- Support pricing strategies

In [0]:
%sql

SELECT
    product,
    SUM(Revenue) AS product_revenue
FROM silver_sales_view
GROUP BY product
ORDER BY product_revenue DESC

product,product_revenue
Laptop,1701.0
Gaming Console,999.98
Tablet,961.1999999999999
Drone,799.5
Smartphone,650.75
Smartwatch,630.9000000000001
Monitor,560.3
Camera,540.0
Headphones,483.96
Printer,190.0


# 4. Region-wise Order Count

### Objective
Calculate total number of orders handled by each region.

### Business Significance
This helps:
- Understand transaction distribution
- Measure operational workload
- Analyze customer activity by region
- Support regional planning

In [0]:
%sql

SELECT
    region,
    COUNT(order_id) AS total_orders
FROM silver_sales_view
GROUP BY region
ORDER BY total_orders DESC

region,total_orders
East,3
North,3
South,2
West,2


# 5. Highest Revenue Store

### Objective
Identify the store generating the highest revenue.

### Business Significance
This helps:
- Benchmark top-performing stores
- Evaluate operational success
- Support expansion and investment planning
- Recognize high-performing locations

In [0]:
%sql

SELECT
    store_id,
    SUM(Revenue) AS total_revenue
FROM silver_sales_view
GROUP BY store_id
ORDER BY total_revenue DESC
LIMIT 1

store_id,total_revenue
101,1701.0


# 6. Monthly Revenue Trend

### Objective
Analyze revenue trend over time for business performance tracking.

### Business Significance
Monthly revenue trends help:
- Monitor business growth
- Identify sales patterns
- Support forecasting
- Detect seasonal trends

In [0]:
%sql

SELECT
    SUBSTRING(timestamp, 1, 7) AS month,
    SUM(Revenue) AS monthly_revenue
FROM silver_sales_view
GROUP BY month
ORDER BY month

month,monthly_revenue
2025-04,7517.59


# 7. High Revenue Orders Detection

### Objective
Identify unusually high-value transactions from retail sales data.

### Business Significance
High-value order analysis helps:
- Detect premium customer activity
- Identify bulk purchases
- Support fraud monitoring
- Improve VIP customer targeting

In [0]:
%sql

SELECT
    order_id,
    product,
    region,
    Revenue
FROM silver_sales_view
WHERE Revenue > 5000
ORDER BY Revenue DESC

order_id,product,region,Revenue


# Month-over-Month Growth (MoM)

## Objective
Analyze monthly revenue growth trend using Spark SQL.

## Business Significance
MoM analysis helps:
- Track business growth
- Identify declining periods
- Monitor revenue trends
- Support forecasting and planning

In [0]:
%sql

SELECT
    SUBSTRING(timestamp, 1, 7) AS month,
    SUM(Revenue) AS monthly_revenue
FROM silver_sales_view
GROUP BY month
ORDER BY month

month,monthly_revenue
2025-04,7517.59


# Average Order Value Per Region

## Objective
Calculate average revenue generated per order in each region.

## Business Significance
This helps:
- Compare customer spending behavior
- Evaluate regional purchasing power
- Support pricing strategy decisions

In [0]:
%sql

SELECT
    region,
    AVG(Revenue) AS avg_order_value
FROM silver_sales_view
GROUP BY region
ORDER BY avg_order_value DESC

region,avg_order_value
North,1013.5
West,741.97
East,717.4666666666666
South,420.375


# Store Performance Segmentation

## Objective
Classify stores into High, Medium, and Low performers based on revenue.

## Business Significance
This helps:
- Identify top-performing stores
- Detect weak-performing outlets
- Support business optimization

In [0]:
%sql

WITH store_revenue AS (

    SELECT
        store_id,
        SUM(Revenue) AS total_revenue
    FROM silver_sales_view
    GROUP BY store_id

)

SELECT
    store_id,
    total_revenue,

    CASE

        WHEN total_revenue >= 700 THEN 'High'
        WHEN total_revenue >= 400 THEN 'Medium'
        ELSE 'Low'

    END AS performance_segment

FROM store_revenue
ORDER BY total_revenue DESC

store_id,total_revenue,performance_segment
101,1701.0,High
108,999.98,High
103,961.1999999999999,High
109,799.5,High
102,650.75,Medium
110,630.9000000000001,Medium
106,560.3,Medium
105,540.0,Medium
104,483.96,Medium
107,190.0,Low


# Top 5 Products by Revenue (Global)

## Objective
Identify globally highest revenue-generating products using SQL ranking.

## Business Significance
This helps:
- Identify best-selling products
- Optimize inventory planning
- Improve product strategy

In [0]:
%sql

SELECT
    product,
    SUM(Revenue) AS total_revenue,
    DENSE_RANK() OVER (
        ORDER BY SUM(Revenue) DESC
    ) AS product_rank

FROM silver_sales_view

GROUP BY product

ORDER BY product_rank

product,total_revenue,product_rank
Laptop,1701.0,1
Gaming Console,999.98,2
Tablet,961.1999999999999,3
Drone,799.5,4
Smartphone,650.75,5
Smartwatch,630.9000000000001,6
Monitor,560.3,7
Camera,540.0,8
Headphones,483.96,9
Printer,190.0,10


# Top 5 Products by Revenue (Region-wise)

## Objective
Rank products within each region using window functions.

## Business Significance
This helps:
- Analyze regional product preferences
- Support localized inventory planning
- Improve regional marketing strategy

In [0]:
%sql

WITH regional_product_revenue AS (

    SELECT
        region,
        product,
        SUM(Revenue) AS total_revenue

    FROM silver_sales_view

    GROUP BY region, product

)

SELECT *

FROM (

    SELECT
        region,
        product,
        total_revenue,

        DENSE_RANK() OVER (
            PARTITION BY region
            ORDER BY total_revenue DESC
        ) AS regional_rank

    FROM regional_product_revenue

)

WHERE regional_rank <= 5

region,product,total_revenue,regional_rank
East,Tablet,961.1999999999999,1
East,Smartwatch,630.9000000000001,2
East,Monitor,560.3,3
North,Laptop,1701.0,1
North,Drone,799.5,2
North,Camera,540.0,3
South,Smartphone,650.75,1
South,Printer,190.0,2
West,Gaming Console,999.98,1
West,Headphones,483.96,2


# Rolling Revenue (7-Day Moving Average)

## Objective
Calculate rolling average revenue trend using SQL window functions.

## Business Significance
Rolling averages help:
- Smooth revenue fluctuations
- Identify trend direction
- Improve forecasting
- Detect unusual patterns

In [0]:
%sql

WITH daily_revenue AS (

    SELECT
        DATE(timestamp) AS order_date,
        SUM(Revenue) AS daily_revenue

    FROM silver_sales_view

    GROUP BY DATE(timestamp)

)

SELECT
    order_date,
    daily_revenue,

    AVG(daily_revenue) OVER (

        ORDER BY order_date

        ROWS BETWEEN 6 PRECEDING
        AND CURRENT ROW

    ) AS rolling_7day_avg

FROM daily_revenue

ORDER BY order_date

order_date,daily_revenue,rolling_7day_avg
2025-04-01,7517.59,7517.59


# Anomaly Detection

## Objective
Identify abnormal revenue days where sales significantly deviate from average revenue.

## Business Significance
Anomaly detection helps:
- Detect unexpected business spikes
- Identify operational issues
- Support fraud monitoring
- Improve business monitoring systems

In [0]:
%sql

WITH daily_revenue AS (

    SELECT
        DATE(timestamp) AS order_date,
        SUM(Revenue) AS daily_revenue

    FROM silver_sales_view

    GROUP BY DATE(timestamp)

),

average_revenue AS (

    SELECT
        AVG(daily_revenue) AS avg_rev

    FROM daily_revenue

)

SELECT
    d.order_date,
    d.daily_revenue,
    a.avg_rev,

    ROUND(
        ABS(d.daily_revenue - a.avg_rev)
        / a.avg_rev * 100,
        2
    ) AS deviation_percent

FROM daily_revenue d
CROSS JOIN average_revenue a

WHERE
    ABS(d.daily_revenue - a.avg_rev)
    / a.avg_rev > 0.30

ORDER BY deviation_percent DESC

order_date,daily_revenue,avg_rev,deviation_percent


# Customer Behavior Proxy Analysis

## Objective
Since customer IDs are unavailable, StoreID is used as a proxy to analyze repeat transaction behavior patterns.

## Business Significance
This helps:
- Understand repeat transaction activity
- Identify highly active stores
- Analyze operational transaction frequency
- Simulate customer behavior analytics

In [0]:
%sql

SELECT
    store_id,
    COUNT(order_id) AS total_orders,
    SUM(Revenue) AS total_revenue,
    AVG(Revenue) AS avg_order_value

FROM silver_sales_view

GROUP BY store_id

ORDER BY total_orders DESC

store_id,total_orders,total_revenue,avg_order_value
105,1,540.0,540.0
108,1,999.98,999.98
109,1,799.5,799.5
104,1,483.96,483.96
103,1,961.1999999999999,961.1999999999999
107,1,190.0,190.0
102,1,650.75,650.75
101,1,1701.0,1701.0
110,1,630.9000000000001,630.9000000000001
106,1,560.3,560.3


# PART D – Orchestration

# D1 – Airflow Theory

## 1. DAG Structure

A DAG (Directed Acyclic Graph) is the core workflow structure in Apache Airflow.

Characteristics:
- Represents task dependencies
- Executes tasks in sequence
- No cyclic dependencies allowed
- Enables pipeline orchestration

Example in this project:
Kafka Ingestion → Bronze Layer → Silver Layer → Gold Layer

---

## 2. Idempotency

Idempotency means a pipeline produces the same output even if executed multiple times.

Importance:
- Prevents duplicate records
- Ensures reliable reruns
- Critical for production pipelines

In this project:
- Delta tables
- Deduplication logic
- Checkpointing
help maintain idempotency.

---

## 3. Backfills

Backfill means running pipelines for historical or missed data periods.

Use Cases:
- Recover failed jobs
- Process delayed data
- Historical data correction

Example:
Reprocessing missed Kafka batches into Bronze/Silver layers.

---

## 4. Retry Mechanisms

Retries automatically rerun failed tasks.

Benefits:
- Handles temporary failures
- Improves pipeline reliability
- Reduces manual intervention

Typical Airflow configuration:
- retry count
- retry delay
- exponential backoff

---

## 5. Sensors

Sensors wait for external events before triggering workflows.

Examples:
- File arrival sensor
- Kafka availability sensor
- Database update sensor

In this project:
Sensors can monitor Kafka topic readiness before triggering streaming jobs.

# D2 – Azure Data Factory (ADF) Pipeline Design

## Objective

Design an orchestration pipeline for:

Kafka → Bronze → Silver → Gold

---

# Proposed Pipeline Flow

## Step 1 – Kafka Ingestion
- Streaming sales events generated using Kafka Producer
- Confluent Cloud acts as streaming broker

Trigger:
- Continuous/Event-based trigger

---

## Step 2 – Bronze Layer

Purpose:
- Store raw streaming data
- Preserve original schema
- Add metadata columns

Technology:
- Spark Structured Streaming
- Delta Lake

Monitoring:
- Streaming query monitoring
- Kafka lag tracking

---

## Step 3 – Silver Layer

Purpose:
- Clean and transform data
- Remove duplicates
- Standardize regions
- Create revenue column

Trigger:
- Trigger after Bronze completion

Monitoring:
- Data quality validation
- Null checks
- Schema validation

---

## Step 4 – Gold Layer

Purpose:
- KPI generation
- Executive analytics
- SQL reporting

Outputs:
- Revenue KPIs
- Store rankings
- Trend analytics

Trigger:
- Scheduled batch trigger

---

# Monitoring Strategy

The pipeline should monitor:
- Kafka lag
- Streaming failures
- Delta write failures
- Schema drift
- Data freshness

Tools:
- Databricks monitoring
- Azure Monitor
- Airflow logs
- ADF monitoring dashboard

---

# Cost Considerations

## Streaming Costs
- Continuous clusters increase compute costs
- Kafka retention increases storage costs

## Optimization Strategies
- Auto-scaling clusters
- Incremental processing
- Delta optimization
- Scheduled cluster shutdown

## Trade-offs
Lower latency improves freshness but increases infrastructure cost.

# PART E – Streaming Design Patterns

---

# 1. Stateful vs Stateless Streaming

## Stateless Streaming

Stateless streaming processes each event independently without maintaining historical information.

Characteristics:
- Faster processing
- Lower memory usage
- No dependency on previous events

Examples:
- Filtering records
- Data format conversion
- Simple transformations

In this project:
Kafka JSON parsing is stateless.

---

## Stateful Streaming

Stateful streaming maintains information across multiple events.

Characteristics:
- Requires memory/state management
- Enables aggregations and windowing
- Supports trend analysis

Examples:
- Running totals
- Window aggregations
- Session analysis

In this project:
Revenue aggregation and rolling averages are stateful operations.

---

# 2. What is Watermarking?

Watermarking is a streaming technique used to handle delayed or late-arriving events.

Purpose:
- Prevent infinite waiting for delayed data
- Manage streaming state efficiently
- Control memory usage

Example:
If watermark is 10 minutes:
- Events arriving later than 10 minutes may be ignored.

Benefits:
- Improves scalability
- Enables reliable window processing
- Prevents excessive state accumulation

---

# 3. Handling Late-Arriving Data

Late-arriving data refers to events reaching the system after expected processing time.

Common Causes:
- Network delays
- Kafka lag
- System failures
- Delayed producers

Handling Strategies:
- Watermarking
- Windowing
- Retry mechanisms
- Dead-letter queues

In enterprise systems:
Late data handling is critical for accurate analytics and reconciliation.

---

# 4. Exactly-Once in Spark + Kafka

Exactly-once processing ensures records are processed only once without duplication.

Spark + Kafka achieve this using:
- Checkpointing
- Offset tracking
- Idempotent writes
- Delta Lake ACID transactions

Workflow:
1. Kafka stores offsets
2. Spark tracks processed batches
3. Delta ensures transactional writes
4. Failed jobs recover safely from checkpoints

Benefits:
- Prevents duplicate records
- Improves data consistency
- Ensures reliable streaming pipelines

# PART F – Vector DB & Semantic Search

# F1 – Theory

---

# 1. What are Embeddings?

Embeddings are numerical vector representations of text, images, or other data.

Purpose:
- Capture semantic meaning
- Convert unstructured text into machine-readable vectors
- Enable similarity comparison

Example:
Product descriptions with similar meanings generate similar embeddings.

Applications:
- Semantic search
- Recommendation systems
- Chatbots
- NLP systems

---

# 2. Why Cosine Similarity Works?

Cosine similarity measures similarity between vectors based on angle rather than magnitude.

Formula:
Cosine Similarity = cosine(angle between vectors)

Why it works:
- Similar text produces vectors pointing in similar directions
- Independent of vector size
- Effective for semantic comparison

Interpretation:
- Value near 1 → highly similar
- Value near 0 → unrelated
- Value near -1 → opposite meaning

---

# 3. Keyword Search vs Semantic Search

## Keyword Search

Searches exact matching words.

Characteristics:
- Fast
- Simple
- Fails on synonyms/context

Example:
Searching “phone” may not match “smartphone”.

---

## Semantic Search

Understands contextual meaning using embeddings.

Characteristics:
- Context-aware
- Handles synonyms
- Better search relevance

Example:
“Gaming device” may match “Gaming Console”.

---

# 4. What is ANN (Approximate Nearest Neighbour)?

ANN is an optimized search technique for finding similar vectors efficiently.

Purpose:
- Speed up similarity search
- Avoid brute-force comparison
- Handle large vector datasets

Applications:
- Vector databases
- Recommendation systems
- Semantic search engines

Popular ANN libraries:
- FAISS
- ChromaDB
- Annoy
- Pinecone

# F2 – Practical Semantic Search

## Objective

Implement semantic product search using:
- Sentence Transformers
- Vector embeddings
- ChromaDB vector database

The system enables context-aware product search instead of simple keyword matching.

In [0]:
product_data = [

    {
        "product_id": 1,
        "product_name": "Gaming Laptop",
        "category": "Electronics",
        "description": "High performance laptop designed for gaming and graphics intensive applications"
    },

    {
        "product_id": 2,
        "product_name": "Wireless Headphones",
        "category": "Audio",
        "description": "Noise cancelling wireless headphones with premium sound quality"
    },

    {
        "product_id": 3,
        "product_name": "Smartphone",
        "category": "Mobile",
        "description": "Android smartphone with high resolution camera and fast processor"
    },

    {
        "product_id": 4,
        "product_name": "Gaming Console",
        "category": "Entertainment",
        "description": "Next generation gaming console with immersive graphics"
    },

    {
        "product_id": 5,
        "product_name": "Smartwatch",
        "category": "Wearable",
        "description": "Fitness tracking smartwatch with heart rate monitoring"
    }

]

In [0]:
import pandas as pd

products_df = pd.DataFrame(product_data)

products_df

,product_id,product_name,category,description
0,1,Gaming Laptop,Electronics,High performance laptop designed for gaming an...
1,2,Wireless Headphones,Audio,Noise cancelling wireless headphones with prem...
2,3,Smartphone,Mobile,Android smartphone with high resolution camera...
3,4,Gaming Console,Entertainment,Next generation gaming console with immersive ...
4,5,Smartwatch,Wearable,Fitness tracking smartwatch with heart rate mo...


In [0]:
%pip install sentence-transformers chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.7/22.7 MB 160.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 177.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.4/426.4 MB 142.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.6/444.6 MB 144.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.1/221.1 MB 146.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.5/188.5 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
from sentence_transformers import SentenceTransformer
import chromadb

In [0]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
product_data = [

    {
        "product_id": 1,
        "product_name": "Gaming Laptop",
        "category": "Electronics",
        "description": "High performance laptop designed for gaming and graphics intensive applications"
    },

    {
        "product_id": 2,
        "product_name": "Wireless Headphones",
        "category": "Audio",
        "description": "Noise cancelling wireless headphones with premium sound quality"
    },

    {
        "product_id": 3,
        "product_name": "Smartphone",
        "category": "Mobile",
        "description": "Android smartphone with high resolution camera and fast processor"
    },

    {
        "product_id": 4,
        "product_name": "Gaming Console",
        "category": "Entertainment",
        "description": "Next generation gaming console with immersive graphics"
    },

    {
        "product_id": 5,
        "product_name": "Smartwatch",
        "category": "Wearable",
        "description": "Fitness tracking smartwatch with heart rate monitoring"
    }

]

In [0]:
import pandas as pd

products_df = pd.DataFrame(product_data)

In [0]:
embeddings = model.encode(
    products_df["description"].tolist()
)

In [0]:
client = chromadb.Client()

collection = client.create_collection(
    name="smartgear_products"
)

In [0]:
for idx, row in products_df.iterrows():

    collection.add(

        ids=[str(row["product_id"])],

        embeddings=[embeddings[idx].tolist()],

        documents=[row["description"]],

        metadatas=[

            {
                "product_name": row["product_name"],
                "category": row["category"]
            }

        ]
    )

In [0]:
def semantic_search(query_text):

    query_embedding = model.encode([query_text])

    results = collection.query(

        query_embeddings=query_embedding.tolist(),

        n_results=3

    )

    return results

In [0]:
results = semantic_search(
    "high performance gaming device"
)

results

{'ids': [['1', '4', '3']],
 'embeddings': None,
 'documents': [['High performance laptop designed for gaming and graphics intensive applications',
   'Next generation gaming console with immersive graphics',
   'Android smartphone with high resolution camera and fast processor']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'category': 'Electronics', 'product_name': 'Gaming Laptop'},
   {'product_name': 'Gaming Console', 'category': 'Entertainment'},
   {'category': 'Mobile', 'product_name': 'Smartphone'}]],
 'distances': [[0.4236328601837158, 0.9142186641693115, 1.2973148822784424]]}

In [0]:
for i in range(len(results["documents"][0])):

    print("\nResult", i + 1)

    print(
        "Product:",
        results["metadatas"][0][i]["product_name"]
    )

    print(
        "Category:",
        results["metadatas"][0][i]["category"]
    )

    print(
        "Description:",
        results["documents"][0][i]
    )


Result 1
Product: Gaming Laptop
Category: Electronics
Description: High performance laptop designed for gaming and graphics intensive applications

Result 2
Product: Gaming Console
Category: Entertainment
Description: Next generation gaming console with immersive graphics

Result 3
Product: Smartphone
Category: Mobile
Description: Android smartphone with high resolution camera and fast processor


# PART G – Mini Case Study

## Scenario

SmartGear leadership observes:
- Revenue mismatch between streaming and batch reports
- Sudden spike in Kafka lag
- Missing records in Gold layer

As a Data Engineering Lead, the following analysis and action plan is proposed.

---

# 1. Possible Root Causes

## A. Revenue Mismatch Between Streaming and Batch Reports

Possible Causes:
- Late-arriving Kafka events
- Duplicate event processing
- Data loss during streaming failures
- Incorrect aggregation logic
- Batch and streaming using different processing windows
- Schema inconsistencies between pipelines

Impact:
- Inaccurate KPI reporting
- Executive dashboard inconsistency
- Business trust issues

---

## B. Sudden Spike in Kafka Lag

Possible Causes:
- Slow Spark consumers
- Insufficient Kafka partitions
- Cluster resource exhaustion
- Streaming job failures
- Large message bursts
- Network bottlenecks

Impact:
- Delayed analytics
- Increased processing latency
- Streaming backlog accumulation

---

## C. Missing Records in Gold Layer

Possible Causes:
- Failed Silver-to-Gold transformations
- Incorrect filtering conditions
- Deduplication removing valid records
- Delta write failures
- Schema evolution issues
- Incomplete checkpoint recovery

Impact:
- Incomplete executive reporting
- Incorrect KPIs
- Data inconsistency across layers

---

# 2. Debugging Strategy

## Step 1 – Validate Kafka Pipeline

Actions:
- Check Kafka topic health
- Verify partition distribution
- Monitor consumer offsets
- Compare produced vs consumed record counts

Tools:
- Confluent Cloud monitoring
- Kafka consumer groups
- Offset tracking dashboards

---

## Step 2 – Validate Bronze Layer

Checks:
- Verify raw event counts
- Compare Kafka offsets with Bronze records
- Inspect schema consistency
- Validate checkpoint recovery

---

## Step 3 – Validate Silver Transformations

Checks:
- Null handling logic
- Duplicate removal rules
- Revenue calculation correctness
- Data type consistency

---

## Step 4 – Validate Gold Aggregations

Checks:
- Aggregation logic
- SQL transformation correctness
- Window function outputs
- Revenue reconciliation with batch dataset

---

## Step 5 – Resource Monitoring

Checks:
- Spark cluster CPU and memory
- Streaming micro-batch duration
- Executor failures
- Shuffle bottlenecks

---

# 3. Monitoring Framework Design

## A. Kafka Monitoring

Metrics:
- Consumer lag
- Topic throughput
- Partition imbalance
- Failed messages

Tools:
- Confluent Cloud dashboard
- Prometheus
- Grafana

---

## B. Streaming Monitoring

Metrics:
- Micro-batch processing time
- Streaming query failures
- Checkpoint health
- Throughput per second

Tools:
- Spark Structured Streaming UI
- Databricks monitoring

---

## C. Data Quality Monitoring

Checks:
- Null percentage
- Duplicate percentage
- Schema drift
- Revenue reconciliation

Tools:
- Great Expectations
- Delta expectations
- Custom validation jobs

---

## D. Gold Layer Monitoring

Metrics:
- KPI freshness
- Missing partitions
- Aggregation consistency
- Report generation failures

---

# Final Recommendation

A production-grade retail data platform should implement:
- End-to-end observability
- Automated alerting
- Schema governance
- Checkpoint recovery
- Data reconciliation pipelines
- Streaming SLA monitoring

This ensures:
- Reliable analytics
- Scalable streaming operations
- Accurate business KPIs
- Enterprise-grade data engineering governance